In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv


In [8]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv")
# If the path errors, copy the exact path from the Add Data panel

print(df.shape)

df.info()
print(df["Machine failure"].value_counts())
print(df.head())

(10000, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(

In [9]:
failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
print(df[failure_cols].sum())

print(df.groupby("Machine failure")[
    ["Air temperature [K]", "Process temperature [K]",
     "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
].mean())

TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64
                 Air temperature [K]  Process temperature [K]  \
Machine failure                                                 
0                         299.973999               309.995570   
1                         300.886431               310.290265   

                 Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  
Machine failure                                                        
0                           1540.260014    39.629655       106.693717  
1                           1496.486726    50.168142       143.781711  


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Remove identifiers and answer-key columns
df2 = df.drop(columns=["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"])

# 2. Turn the Type letter into numbers (L/M/H = low/medium/high quality)
df2["Type"] = df2["Type"].map({"L": 0, "M": 1, "H": 2})

# 3. Separate inputs (X) from the answer (y)
X = df2.drop(columns=["Machine failure"])
y = df2["Machine failure"]

# 4. Stratified split: both sets keep the same 3.4% failure ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 5. Train, with failures weighted more heavily
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

# 6. Evaluate properly
pred = model.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))

[[1928    4]
 [  31   37]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.90      0.54      0.68        68

    accuracy                           0.98      2000
   macro avg       0.94      0.77      0.83      2000
weighted avg       0.98      0.98      0.98      2000

ROC-AUC: 0.9680991962002192


In [11]:
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

# 1. New columns built from existing sensors
def add_features(d):
    d = d.copy()
    d["power"] = d["Torque [Nm]"] * d["Rotational speed [rpm]"] * 2 * np.pi / 60
    d["temp_diff"] = d["Process temperature [K]"] - d["Air temperature [K]"]
    d["wear_torque"] = d["Tool wear [min]"] * d["Torque [Nm]"]
    return d

X_train2 = add_features(X_train)
X_test2 = add_features(X_test)

# 2. Retrain with the new features
model2 = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=42, n_jobs=-1)

# 3. Out-of-fold probabilities on TRAINING data only
probs_cv = cross_val_predict(model2, X_train2, y_train, cv=5,
                             method="predict_proba")[:, 1]

# 4. Lowest-risk threshold that still gives recall >= 0.90
prec, rec, thr = precision_recall_curve(y_train, probs_cv)
ok = np.where(rec[:-1] >= 0.90)[0]
best = ok[np.argmax(prec[:-1][ok])]
threshold = thr[best]
print("Chosen threshold:", threshold)

# 5. Fit on full train, evaluate ONCE on test
model2.fit(X_train2, y_train)
probs_test = model2.predict_proba(X_test2)[:, 1]
pred2 = (probs_test >= threshold).astype(int)

print(confusion_matrix(y_test, pred2))
print(classification_report(y_test, pred2))
print("ROC-AUC:", roc_auc_score(y_test, probs_test))

# 6. Which sensors mattered
print(sorted(zip(model2.feature_importances_, X_train2.columns), reverse=True))

Chosen threshold: 0.05
[[1840   92]
 [   6   62]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      1932
           1       0.40      0.91      0.56        68

    accuracy                           0.95      2000
   macro avg       0.70      0.93      0.77      2000
weighted avg       0.98      0.95      0.96      2000

ROC-AUC: 0.9743941054682743
[(np.float64(0.20712912620520033), 'Rotational speed [rpm]'), (np.float64(0.1910370982137949), 'Torque [Nm]'), (np.float64(0.18168634609886858), 'power'), (np.float64(0.14300238015198652), 'Tool wear [min]'), (np.float64(0.11511565327966683), 'wear_torque'), (np.float64(0.08882705718271751), 'temp_diff'), (np.float64(0.04230698395853075), 'Air temperature [K]'), (np.float64(0.024356020018514685), 'Process temperature [K]'), (np.float64(0.00653933489071992), 'Type')]


In [12]:
import joblib, sklearn
print(sklearn.__version__)   # note this number

joblib.dump({
    "model": model2,
    "threshold": float(threshold),
    "features": list(X_train2.columns)
}, "model.joblib")

1.6.1


['model.joblib']

In [13]:
for t in [0.05, 0.10, 0.20, 0.30, 0.50]:
    p = (probs_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    print(f"t={t:.2f}  recall={tp/(tp+fn):.2f}  precision={tp/(tp+fp):.2f}  missed={fn}  false_alarms={fp}")

t=0.05  recall=0.91  precision=0.40  missed=6  false_alarms=92
t=0.10  recall=0.90  precision=0.55  missed=7  false_alarms=49
t=0.20  recall=0.84  precision=0.81  missed=11  false_alarms=13
t=0.30  recall=0.82  precision=0.92  missed=12  false_alarms=5
t=0.50  recall=0.75  precision=0.96  missed=17  false_alarms=2
